# Module 1: Data Preparation for Vertex AI Tuning (Native BigQuery)

In this notebook, we will prepare the synthetic cybersecurity data generated by BigQuery ML (stored in the `sft_data` and `lora_data` tables).

Instead of downloading the data into local memory using `pandas` or `sklearn` (which does not scale), we will utilize **Google Cloud native capabilities**:
1. **BigFrames**: To interactively preview the BigQuery tables using a Pandas-like API that runs securely in the BigQuery engine.
2. **BigQuery EXPORT DATA**: To split our dataset and automatically export the data directly to Google Cloud Storage (GCS) in the required `JSONL` format for Vertex AI instruction tuning.

This in-place processing architecture ensures scalability.

In [ ]:
.pip install bigframes google-cloud-bigquery python-dotenv -q

## 1. Setup and Authentication

In [ ]:
import os
from google.cloud import bigquery
import bigframes.pandas as bpd
from dotenv import load_dotenv

# Load environment variables
load_dotenv()  # Ensure you have a .env file in the same directory

PROJECT_ID = os.getenv("PROJECT_ID", "YOUR_PROJECT_ID")
LOCATION = os.getenv("LOCATION", "global") 
DATASET_ID = os.getenv("DATASET_ID", "tuning")

# You should define a GCS bucket in your .env or replace it here
STAGING_BUCKET = os.getenv("STAGING_BUCKET", "YOUR_GCS_BUCKET_NAME")
if STAGING_BUCKET.startswith("gs://"):
    STAGING_BUCKET = STAGING_BUCKET.replace("gs://", "")

# Initialize Native BigQuery & BigFrames
bq_client = bigquery.Client(project=PROJECT_ID)
bpd.options.bigquery.project = PROJECT_ID
bpd.options.bigquery.location = LOCATION

print(f"Project ID: {PROJECT_ID}")
print(f"Data Engine Location: {LOCATION}")
print(f"Staging Bucket: gs://{STAGING_BUCKET}")

## 2. Preview Data with BigFrames

BigFrames allows us to interact with BigQuery datasets similar to a Pandas DataFrame, but the processing is pushed down into BigQuery's parallel compute engine.

In [ ]:
print("Previewing SFT Data natively via BigQuery with BigFrames:")

# This does NOT pull the whole dataset into local memory.
df_sft = bpd.read_gbq(f"{PROJECT_ID}.{DATASET_ID}.sft_data")
df_sft.head(3)

## 3. Split & Export Data Natively to GCS (JSONL)

Vertex AI Tuning expects instruction-tuning data to be formatted as JSONL where each line has a `messages` array.
Since our BigQuery column is already named `messages` and holds JSON, we can use the BigQuery `EXPORT DATA` SQL statement to directly dump the data into GCS in exactly the format Vertex AI needs, without any intermediate Python processing.

## 4. URIs for Vertex AI

We will use these URIs in **Module 2** (LoRA) and **Module 3** (Full SFT) to feed the instruction data into our Vertex AI Supervised Fine-Tuning pipelines.

In [ ]:
print("\n--- COPY THESE URIs FOR THE NEXT NOTEBOOKS ---")
print(f"SFT_TRAIN_URI = '{sft_train_uri}'")
print(f"SFT_EVAL_URI = '{sft_eval_uri}'")
print(f"LORA_TRAIN_URI = '{lora_train_uri}'")
print(f"LORA_EVAL_URI = '{lora_eval_uri}'")